# Thực nghiệm Khôi phục Vùng Ảnh Grayscale bị Khuyết (Gauss–Seidel trên Ma trận Thưa CSR)

**Học phần:** Phương pháp tính — Hướng B (Giải quyết bài toán ứng dụng kết hợp khảo sát hội tụ thực nghiệm)  
**Nhóm thực hiện:** An (Nhóm trưởng) – Giáp – Sơn – Hiếu  
**Điều phối Notebook:** Hiếu (Đánh giá, TN2 & Biên tập PDF)

### Mục tiêu Notebook
Notebook này điều phối trọn bộ quy trình thực nghiệm để đảm bảo khả năng tái lập 100% khi chạy **`Kernel -> Restart & Run All`**:
1. Ghi nhận cấu hình môi trường thực nghiệm (CPU, RAM, phiên bản thư viện).
2. Chạy kiểm thử tự động bộ test chuẩn $2 \times 2$ xác nhận 5 module trong `src/` khớp nhau.
3. **Thí nghiệm 1 (TN1 — Giáp chủ trì):** Ảnh hưởng của hình dạng & kích thước vùng khuyết (`2x50`, `10x10`, `30x30`).
4. **Thí nghiệm 2 (TN2 — Hiếu chủ trì):** Ảnh hưởng của giá trị khởi tạo $x^{(0)}$ (`zero`, `random` với `seed=42`, `boundary_mean`).
5. **Thí nghiệm 3 (TN3 — Giáp chủ trì):** Ảnh hưởng của ngưỡng dừng $TOL \in \{10^{-2}, 10^{-3}, 10^{-4}, 10^{-5}\}$ và kiểm tra hội tụ cho mask $30 \times 30$ ở $TOL = 10^{-5}$.
6. **Kiểm chứng bổ sung trên Ảnh 2 (Sơn & Hiếu):** Đối chiếu cấu hình chuẩn trên ảnh nhiều chi tiết cạnh.
7. Xuất đầy đủ bảng CSV (`results/tables/`), ảnh khôi phục (`results/restored_images/`) và biểu đồ (`results/plots/`).

## 1. Thiết lập môi trường & Nạp các module trong `src/`

In [ ]:
import csv
import os
import platform
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import PIL
import scipy

# Thêm thư mục gốc dự án vào sys.path (khi notebook nằm trong experiments/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Tạo sẵn các thư mục đầu ra chuẩn
DATA_ORIGINAL_DIR = PROJECT_ROOT / "data" / "original"
DATA_MASKS_DIR = PROJECT_ROOT / "data" / "masks"
RESULTS_IMAGES_DIR = PROJECT_ROOT / "results" / "restored_images"
RESULTS_TABLES_DIR = PROJECT_ROOT / "results" / "tables"
RESULTS_PLOTS_DIR = PROJECT_ROOT / "results" / "plots"

for d in (RESULTS_IMAGES_DIR, RESULTS_TABLES_DIR, RESULTS_PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Nạp 5 module chuẩn của nhóm (sẽ sẵn sàng sau khi cả 4 thành viên hoàn thiện ở Tuần 2)
from src.image_processing import load_grayscale, restore_pixels, save_image
from src.mask import create_scratch_mask, create_square_mask
from src.laplace_model import build_system, make_initial_guess, solve_reference
from src.gauss_seidel import gauss_seidel
from src.evaluation import (
    benchmark_solver,
    compute_mse_psnr,
    compute_relative_residual,
    compute_solution_error,
)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

In [ ]:
# Ghi nhận thông số môi trường thực nghiệm (Checklist #5 — đưa vào Mục 3.4 Báo cáo)
def print_environment_info():
    ram_info = "Không xác định (cài psutil để hiển thị tự động)"
    try:
        import psutil
        ram_gb = psutil.virtual_memory().total / (1024 ** 3)
        ram_info = f"{ram_gb:.2f} GB"
    except ImportError:
        if platform.system() == "Windows":
            import ctypes
            class MEMORYSTATUSEX(ctypes.Structure):
                _fields_ = [
                    ("dwLength", ctypes.c_ulong),
                    ("dwMemoryLoad", ctypes.c_ulong),
                    ("ullTotalPhys", ctypes.c_ulonglong),
                    ("ullAvailPhys", ctypes.c_ulonglong),
                    ("ullTotalPageFile", ctypes.c_ulonglong),
                    ("ullAvailPageFile", ctypes.c_ulonglong),
                    ("ullTotalVirtual", ctypes.c_ulonglong),
                    ("ullAvailVirtual", ctypes.c_ulonglong),
                    ("ullAvailExtendedVirtual", ctypes.c_ulonglong),
                ]
            stat = MEMORYSTATUSEX()
            stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
            if ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat)):
                ram_info = f"{stat.ullTotalPhys / (1024 ** 3):.2f} GB"

    print("=== THÔNG SỐ MÔI TRƯỜNG THỰC NGHIỆM ===")
    print(f"Hệ điều hành : {platform.system()} {platform.release()} ({platform.machine()})")
    print(f"CPU          : {platform.processor() or 'N/A'} ({os.cpu_count()} luồng)")
    print(f"RAM          : {ram_info}")
    print(f"Python       : {sys.version.split()[0]}")
    print(f"NumPy        : {np.__version__}")
    print(f"SciPy        : {scipy.__version__}")
    print(f"Pillow       : {PIL.__version__}")
    print(f"Matplotlib   : {matplotlib.__version__}")

print_environment_info()

## 2. Kiểm thử tự động bộ test chuẩn $2 \times 2$

Xác nhận cả 5 module trong `src/` tuân thủ tuyệt đối quy ước mask (`0 = đã biết, 1 = bị khuyết`), thứ tự ẩn số `unknown_pixels = np.argwhere(mask == 1)` và cho nghiệm đúng $x^* \approx [105.4167, 137.0833, 84.5833, 122.9167]^T$.

In [ ]:
from tests.test_2x2 import run_test_2x2

run_test_2x2()

## 3. Hàm điều phối thực nghiệm dùng chung & Tiện ích xuất CSV / Hình ảnh

- `time.perf_counter()` **chỉ bọc quanh** `gauss_seidel`, lặp **5 lần** trên cùng dữ liệu & $x^{(0)}$ và lấy **trung vị (median)**.
- Luôn truyền tường minh `tol=tol, max_iter=max_iter` (tránh lỗi dùng nhầm `tol=1e-6` mặc định ở TN3).
- `MSE_mask` và `PSNR_mask` chỉ tính trên vùng `mask == 1` trực tiếp trên mảng `float64`.

In [ ]:
def run_single_experiment(
    image: np.ndarray,
    mask: np.ndarray,
    init_mode: str = "boundary_mean",
    tol: float = 1e-4,
    max_iter: int = 10000,
    repeats: int = 5,
    seed: int = 42,
    label: str = "",
) -> dict:
    """Chạy 1 cấu hình thực nghiệm đầy đủ theo pipeline chuẩn của dự án."""
    # 1. Dựng hệ phương trình thưa CSR (chỉ gọi build_system 1 lần để cố định thứ tự unknown_pixels)
    A_csr, b, unknown_pixels = build_system(image, mask)
    n_unknowns = len(unknown_pixels)

    # 2. Nghiệm tham chiếu trực tiếp spsolve (chỉ dùng để kiểm chứng sai số giải hệ e_ref)
    x_ref = solve_reference(A_csr, b)

    # 3. Khởi tạo x0
    x0 = make_initial_guess(image, mask, unknown_pixels, mode=init_mode, seed=seed)

    # 4. Đo thời gian thực thi riêng hàm gauss_seidel (lặp 5 lần lấy trung vị)
    bench = benchmark_solver(
        gauss_seidel,
        A_csr,
        b,
        x0=x0,
        tol=tol,
        max_iter=max_iter,
        repeats=repeats,
    )
    x_gs, iters, residual_history, converged = bench["result"]
    median_time = bench["median_time"]

    # 5. Ghép nghiệm vào bản sao ảnh gốc (float64)
    restored = restore_pixels(image, mask, unknown_pixels, x_gs)

    # 6. Tính 2 nhóm chỉ số tách bạch: Độ chính xác giải hệ & Chất lượng ảnh trên vùng mask
    r_rel = compute_relative_residual(A_csr, b, x_gs)
    e_ref = compute_solution_error(x_gs, x_ref)
    img_metrics = compute_mse_psnr(image, restored, mask)

    return {
        "label": label,
        "n_unknowns": n_unknowns,
        "init_mode": init_mode,
        "tol": tol,
        "max_iter": max_iter,
        "iterations": iters,
        "converged": converged,
        "median_time_s": median_time,
        "r_rel": r_rel,
        "e_ref": e_ref,
        "mse_mask": img_metrics["mse"],
        "psnr_mask_db": img_metrics["psnr"],
        "residual_history": residual_history,
        "restored": restored,
    }


def save_summary_csv(records: list, csv_path: Path) -> None:
    """Xuất danh sách kết quả thực nghiệm (bỏ các mảng lớn) ra file CSV chuẩn."""
    exclude_keys = {"residual_history", "restored"}
    fieldnames = [k for k in records[0].keys() if k not in exclude_keys]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for rec in records:
            writer.writerow({k: rec[k] for k in fieldnames})
    print(f"Đã lưu bảng kết quả: {csv_path}")


def plot_restoration_triplet(
    original: np.ndarray,
    mask: np.ndarray,
    restored: np.ndarray,
    title: str,
    save_path: Path,
) -> None:
    """Vẽ đối chiếu 3 khung hình: Ảnh gốc | Ảnh bị khuyết | Ảnh khôi phục."""
    corrupted = original.copy()
    corrupted[mask == 1] = 0.0

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, img, subtitle in zip(
        axes,
        [original, corrupted, restored],
        ["Ảnh gốc", "Ảnh bị khuyết (Mask)", "Ảnh khôi phục (Gauss–Seidel)"],
    ):
        ax.imshow(np.clip(np.rint(img), 0, 255), cmap="gray", vmin=0, vmax=255)
        ax.set_title(subtitle)
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu hình minh họa: {save_path}")

## 4. Nạp bộ dữ liệu ảnh chuẩn hóa (256 × 256) và các Mask cố định

- Ảnh gốc (do Sơn chuẩn bị): `data/original/image1_smooth.png`, `data/original/image2_edges.png`.
- Mask cố định (do Sơn tạo & lưu `.npy` ở Tuần 2): `mask_scratch_2x50.npy`, `mask_sq_10x10.npy`, `mask_sq_30x30.npy`.

In [ ]:
IMG1_PATH = DATA_ORIGINAL_DIR / "image1_smooth.png"
IMG2_PATH = DATA_ORIGINAL_DIR / "image2_edges.png"

MASK_SCRATCH_PATH = DATA_MASKS_DIR / "mask_scratch_2x50.npy"
MASK_SQ10_PATH = DATA_MASKS_DIR / "mask_sq_10x10.npy"
MASK_SQ30_PATH = DATA_MASKS_DIR / "mask_sq_30x30.npy"

image1 = load_grayscale(str(IMG1_PATH))
image2 = load_grayscale(str(IMG2_PATH))

mask_scratch_2x50 = np.load(MASK_SCRATCH_PATH)
mask_sq_10x10 = np.load(MASK_SQ10_PATH)
mask_sq_30x30 = np.load(MASK_SQ30_PATH)

print("Ảnh 1:", image1.shape, image1.dtype)
print("Ảnh 2:", image2.shape, image2.dtype)
print(
    "Số ẩn của các mask:",
    f"2x50={int(mask_scratch_2x50.sum())},",
    f"10x10={int(mask_sq_10x10.sum())},",
    f"30x30={int(mask_sq_30x30.sum())}",
)

## 5. Thí nghiệm 1 (TN1) — Ảnh hưởng của Hình dạng & Kích thước vùng khuyết

- **Phụ trách chính:** Giáp  
- **Biến khảo sát:** Vết xước `2x50` ($n=100$), Vuông `10x10` ($n=100$), Vuông `30x30` ($n=900$).  
- **Cố định:** Ảnh 1 (`image1_smooth.png`), `init_mode="boundary_mean"`, `TOL = 1e-4`, `max_iter = 10000`.

In [ ]:
tn1_configs = [
    ("scratch_2x50", mask_scratch_2x50),
    ("square_10x10", mask_sq_10x10),
    ("square_30x30", mask_sq_30x30),
]

tn1_results = []
for name, m in tn1_configs:
    res = run_single_experiment(
        image=image1,
        mask=m,
        init_mode="boundary_mean",
        tol=1e-4,
        max_iter=10000,
        repeats=5,
        label=name,
    )
    tn1_results.append(res)
    save_image(res["restored"], str(RESULTS_IMAGES_DIR / f"TN1_{name}.png"))
    plot_restoration_triplet(
        image1,
        m,
        res["restored"],
        title=f"TN1 — {name} (iters={res['iterations']}, PSNR={res['psnr_mask_db']:.2f} dB)",
        save_path=RESULTS_IMAGES_DIR / f"TN1_{name}_compare.png",
    )

save_summary_csv(tn1_results, RESULTS_TABLES_DIR / "TN1_summary.csv")

# Vẽ đồ thị hội tụ TN1 (trục y thang log10)
plt.figure(figsize=(8, 5))
for res in tn1_results:
    plt.semilogy(
        range(1, len(res["residual_history"]) + 1),
        res["residual_history"],
        label=f"{res['label']} (n={res['n_unknowns']}, iters={res['iterations']})",
    )
plt.axhline(1e-4, color="gray", linestyle="--", label="TOL = 1e-4")
plt.xlabel("Số vòng lặp (k)")
plt.ylabel("Phần dư tương đối r_rel (thang log)")
plt.title("TN1: Tốc độ hội tụ theo hình dạng và kích thước vùng khuyết")
plt.legend()
plt.grid(True, which="both", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.savefig(RESULTS_PLOTS_DIR / "TN1_convergence.png", dpi=200)
plt.show()

## 6. Thí nghiệm 2 (TN2) — Ảnh hưởng của Giá trị khởi tạo $x^{(0)}$

- **Phụ trách chính:** Hiếu  
- **Biến khảo sát:** `zero`, `random` (`seed=42`), `boundary_mean`.  
- **Cố định:** Ảnh 1 (`image1_smooth.png`), Mask vuông `10x10`, `TOL = 1e-4`, `max_iter = 10000`.

In [ ]:
tn2_modes = ["zero", "random", "boundary_mean"]

tn2_results = []
for mode in tn2_modes:
    res = run_single_experiment(
        image=image1,
        mask=mask_sq_10x10,
        init_mode=mode,
        tol=1e-4,
        max_iter=10000,
        repeats=5,
        seed=42,
        label=mode,
    )
    tn2_results.append(res)
    save_image(res["restored"], str(RESULTS_IMAGES_DIR / f"TN2_{mode}.png"))

save_summary_csv(tn2_results, RESULTS_TABLES_DIR / "TN2_summary.csv")

# Vẽ đồ thị hội tụ TN2
plt.figure(figsize=(8, 5))
for res in tn2_results:
    plt.semilogy(
        range(1, len(res["residual_history"]) + 1),
        res["residual_history"],
        label=f"x0 = {res['label']} (iters={res['iterations']})",
    )
plt.axhline(1e-4, color="gray", linestyle="--", label="TOL = 1e-4")
plt.xlabel("Số vòng lặp (k)")
plt.ylabel("Phần dư tương đối r_rel (thang log)")
plt.title("TN2: Tốc độ hội tụ theo phương thức khởi tạo x^(0) (Mask 10x10)")
plt.legend()
plt.grid(True, which="both", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.savefig(RESULTS_PLOTS_DIR / "TN2_convergence.png", dpi=200)
plt.show()

## 7. Thí nghiệm 3 (TN3) — Ảnh hưởng của Ngưỡng dừng $TOL$

- **Phụ trách chính:** Giáp  
- **Biến khảo sát:** $TOL \in \{10^{-2}, 10^{-3}, 10^{-4}, 10^{-5}\}$ (truyền tường minh `tol=current_tol`).  
- **Cố định:** Ảnh 1 (`image1_smooth.png`), Mask vuông `10x10`, `init_mode="boundary_mean"`, `max_iter = 10000`.  
- **Kiểm tra bổ sung (Checklist #4):** Kiểm tra sớm xem `max_iter = 10000` có đủ hội tụ cho mask `30x30` ở `TOL = 1e-5` hay không.

In [ ]:
tn3_tols = [1e-2, 1e-3, 1e-4, 1e-5]

tn3_results = []
for current_tol in tn3_tols:
    res = run_single_experiment(
        image=image1,
        mask=mask_sq_10x10,
        init_mode="boundary_mean",
        tol=current_tol,
        max_iter=10000,
        repeats=5,
        label=f"TOL={current_tol:.0e}",
    )
    tn3_results.append(res)
    save_image(res["restored"], str(RESULTS_IMAGES_DIR / f"TN3_tol_{current_tol:.0e}.png"))

save_summary_csv(tn3_results, RESULTS_TABLES_DIR / "TN3_summary.csv")

# Kiểm tra ngưỡng hội tụ cho mask 30x30 ở TOL = 1e-5 (Checklist #4)
check_30x30 = run_single_experiment(
    image=image1,
    mask=mask_sq_30x30,
    init_mode="boundary_mean",
    tol=1e-5,
    max_iter=10000,
    repeats=1,
    label="check_30x30_1e-5",
)
print(
    f"[Kiểm tra Checklist #4] Mask 30x30 @ TOL=1e-5: "
    f"converged={check_30x30['converged']}, iters={check_30x30['iterations']}, "
    f"r_rel={check_30x30['r_rel']:.3e}"
)

# Vẽ đồ thị TN3: Số vòng lặp & PSNR_mask theo -log10(TOL)
neg_log_tols = [-np.log10(r["tol"]) for r in tn3_results]
iters_list = [r["iterations"] for r in tn3_results]
psnr_list = [r["psnr_mask_db"] for r in tn3_results]
eref_list = [r["e_ref"] for r in tn3_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(neg_log_tols, iters_list, marker="o", linewidth=2)
ax1.set_xlabel("-log10(TOL)")
ax1.set_ylabel("Số vòng lặp hội tụ")
ax1.set_title("Số vòng lặp theo ngưỡng dừng TOL")
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.plot(neg_log_tols, psnr_list, marker="s", color="tab:green", linewidth=2, label="PSNR_mask (dB)")
ax2.set_xlabel("-log10(TOL)")
ax2.set_ylabel("PSNR_mask (dB)", color="tab:green")
ax2.set_title("Bão hòa PSNR_mask vs. Giảm sai số đại số e_ref")
ax2.grid(True, linestyle=":", alpha=0.6)

ax2_twin = ax2.twinx()
ax2_twin.semilogy(neg_log_tols, eref_list, marker="^", color="tab:red", linestyle="--", label="e_ref (log)")
ax2_twin.set_ylabel("Sai số đại số e_ref (thang log)", color="tab:red")
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="best")

plt.tight_layout()
plt.savefig(RESULTS_PLOTS_DIR / "TN3_psnr.png", dpi=200)
plt.show()

## 8. Kiểm chứng bổ sung trên Ảnh 2 (nhiều chi tiết cạnh sắc nét)

- **Phụ trách:** Sơn & Hiếu  
- Chạy lại cấu hình chuẩn (`mask_sq_10x10`, `init_mode="boundary_mean"`, `TOL = 1e-4`) trên `image2_edges.png` và đối chiếu trực tiếp với Ảnh 1.

In [ ]:
if "tn1_results" in globals():
    img1_std = dict(next(r for r in tn1_results if r["label"] == "square_10x10"))
    img1_std["label"] = "image1_smooth_10x10"
else:
    img1_std = run_single_experiment(
        image=image1,
        mask=mask_sq_10x10,
        init_mode="boundary_mean",
        tol=1e-4,
        max_iter=10000,
        repeats=5,
        label="image1_smooth_10x10",
    )
img2_std = run_single_experiment(
    image=image2,
    mask=mask_sq_10x10,
    init_mode="boundary_mean",
    tol=1e-4,
    max_iter=10000,
    repeats=5,
    label="image2_edges_10x10",
)

save_summary_csv([img1_std, img2_std], RESULTS_TABLES_DIR / "image2_verification.csv")
save_image(img2_std["restored"], str(RESULTS_IMAGES_DIR / "image2_square_10x10.png"))
plot_restoration_triplet(
    image2,
    mask_sq_10x10,
    img2_std["restored"],
    title=f"Kiểm chứng Ảnh 2 — 10x10 (iters={img2_std['iterations']}, PSNR={img2_std['psnr_mask_db']:.2f} dB)",
    save_path=RESULTS_IMAGES_DIR / "image2_square_10x10_compare.png",
)

## 9. Checklist rà soát trước khi xuất số liệu vào Báo cáo (Tuần 4–5)

- [ ] Đã kiểm tra `Kernel -> Restart & Run All` chạy thông suốt không lỗi.
- [ ] Đã đối chiếu toàn bộ số liệu trong `results/tables/TN1_summary.csv`, `TN2_summary.csv`, `TN3_summary.csv`, `image2_verification.csv` trước khi viết nhận xét ở Mục 4 và Mục 5.
- [ ] Đã kiểm tra `max_iter = 10000` đủ hội tụ cho mask `30x30` ở `TOL = 1e-5`.